In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q11/annotated/checkpoints/pre_cell_3.pickle

In [ ]:
Out.clear()

In [ ]:
%%RecordEvent
%%time
### cell 3 ###


partsupp_filtered = partsupp.loc[:, ["PS_PARTKEY", "PS_SUPPKEY"]]
partsupp_filtered["TOTAL_COST"] = (
    partsupp["PS_SUPPLYCOST"] * partsupp["PS_AVAILQTY"]
)
supplier_filtered = supplier.loc[:, ["S_SUPPKEY", "S_NATIONKEY"]]
ps_supp_merge = partsupp_filtered.merge(
    supplier_filtered, left_on="PS_SUPPKEY", right_on="S_SUPPKEY", how="inner"
)
ps_supp_merge = ps_supp_merge.loc[:, ["PS_PARTKEY", "S_NATIONKEY", "TOTAL_COST"]]
nation_filtered = nation[(nation["N_NAME"] == "GERMANY")]
nation_filtered = nation_filtered.loc[:, ["N_NATIONKEY"]]
ps_supp_n_merge = ps_supp_merge.merge(
    nation_filtered, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
ps_supp_n_merge = ps_supp_n_merge.loc[:, ["PS_PARTKEY", "TOTAL_COST"]]
sum_val = ps_supp_n_merge["TOTAL_COST"].sum() * 0.0001
total = ps_supp_n_merge.groupby(["PS_PARTKEY"], as_index=False, sort=False).agg(
    VALUE=pd.NamedAgg(column="TOTAL_COST", aggfunc="sum")
)
total = total[total["VALUE"] > sum_val]
total = total.sort_values("VALUE", ascending=False)


In [ ]:
%%RecordEvent
orig_output = Out.get(5)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q11/annotated/checkpoints/post_cell_3.pickle